In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [43]:
!pip install sklearn-pandas pandas scikit-learn

In [44]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn_pandas import DataFrameMapper

URL_BASE = "https://raw.githubusercontent.com/FranciscoAmbort/grupo13_castellano_ambort_ia_2026/main/entrega_3/data"

url_train = f"{URL_BASE}/solicitudes_train.csv"
url_test = f"{URL_BASE}/solicitudes_test.csv"
url_comp = f"{URL_BASE}/comportamiento_bancario.csv"

# 2. Carga directa optimizada
df_sol_train = pd.read_csv(url_train)
df_sol_test = pd.read_csv(url_test)
df_comportamiento = pd.read_csv(url_comp)

In [45]:
# En estos intentos de regresion logistica cruzamos las tablas de solicitudes_train y comportamiento en base al ID_Usuario, promediando(.mean()) de los datos de un usuario, luego estandarizamos las variables y:
#-submission_monto_ascendente_binario: La variable a predecir es binaria 1(aprobado)o 0(rechazado) y a su vez, los que tienen valor 1 los ordenamos de manera ascendente por el monto solicitado con el objetivo de otorgar prestamos a mas gente, es decir, diversificar el riesgo.
#Esta solucion no funciono(54.000 score en kaggle) y se debe a que el modelo de predicción binaria trata como iguales a clientes de alto y bajo riesgo crediticio al ignorar sus probabilidades reales de pago. (Un cliente con prob_pago de 0,55 es igual que uno de 0,97)
#-submission_predict_proba: ordena por probabilidad descendente de pago, obtuvimos muy buenos resultados con esta regresion logistica simple. (162.000 en kaggle)

# Intentos Regresion Logistica


In [46]:
df_banco_sin_mes = df_comportamiento.drop(columns=['Mes_Referencia'])
comportamiento_resumen = df_banco_sin_mes.groupby('ID_Usuario').mean().reset_index()


In [47]:
#Eliminamos la Fecha_Solicitud
df_sol_train_limpio = df_sol_train.drop(columns=['Fecha_Solicitud'])
df_sol_train_limpio.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Estado_Final
0,1,12698,7900,0.2333,4461,Consumo,1
1,2,20744,17400,0.2507,3889,Vivienda,1
2,3,6830,8600,0.2368,4035,Vehiculo,0
3,4,5722,13300,0.2826,4018,Vehiculo,1
4,5,24719,6600,0.2314,3357,Otros,1


In [48]:
df_sol_comp = df_sol_train_limpio.merge(comportamiento_resumen, on= 'ID_Usuario', how= 'left')


In [49]:
df_sol_comp = df_sol_comp.drop(columns=['ID_Usuario'])
df_sol_comp.head()

,ID_Solicitud,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Estado_Final,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,1,7900,0.2333,4461,Consumo,1,3567.364783,4.826087
1,2,17400,0.2507,3889,Vivienda,1,6070.002273,8.590909
2,3,8600,0.2368,4035,Vehiculo,0,168.037273,19.590909
3,4,13300,0.2826,4018,Vehiculo,1,7564.528696,3.043478
4,5,6600,0.2314,3357,Otros,1,3504.671304,2.478261


In [50]:
sample_sol_comp = df_sol_comp.sample(5, random_state=42)
sample_sol_comp

,ID_Solicitud,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Estado_Final,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
20440,20441,14500,0.2076,5300,Vivienda,1,4835.212917,16.166667
1090,1091,3100,0.2858,776,Consumo,1,1438.028182,1.000000
4762,4763,10200,0.3030,2081,Consumo,1,5136.600455,4.818182
12381,12382,11600,0.2752,3982,Otros,1,7179.826957,6.478261
15112,15113,19900,0.2708,3336,Salud,1,7562.219167,3.500000


In [51]:
train1, validation1 = train_test_split(df_sol_comp, test_size=0.2, random_state=42)

In [52]:
mapper_sol_comp = DataFrameMapper([
    (['Monto_Solicitado'], [StandardScaler()]),
    (['Tasa_Interes'], [StandardScaler()]),
    (['Ingreso_Mensual'], [StandardScaler()]),
    (['Motivo_Prestamo'], [OneHotEncoder()]),
    (['Saldo_Promedio_Mensual'], [StandardScaler()]),
    (['Dias_Atraso_Otras_Deudas'], [StandardScaler()])
])

mapper_sol_comp.fit(train1)
mapper_sol_comp.transform(sample_sol_comp)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


array([[ 0.0194628 , -1.59935254,  0.43706185,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  1.        ,
        -0.12120405,  1.10181234],
       [-0.85588889,  0.81292189, -1.37076513,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        -0.78868554, -1.20895513],
       [-0.31071371,  1.34349887, -0.84927658,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        -0.0619872 , -0.62722346],
       [-0.20321438,  0.4859384 , -0.08962161,  0.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.33946753, -0.37429665],
       [ 0.43410308,  0.3502094 , -0.34776843,  0.        ,  0.        ,
         0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.41460024, -0.82805939]])

In [53]:
mapper_sol_comp.transformed_names_

['Monto_Solicitado',
 'Tasa_Interes',
 'Ingreso_Mensual',
 'Motivo_Prestamo_0',
 'Motivo_Prestamo_1',
 'Motivo_Prestamo_2',
 'Motivo_Prestamo_3',
 'Motivo_Prestamo_4',
 'Motivo_Prestamo_5',
 'Motivo_Prestamo_6',
 'Saldo_Promedio_Mensual',
 'Dias_Atraso_Otras_Deudas']

In [54]:
lr_model_sol_comp1 = Pipeline([
    ('mapper', mapper_sol_comp),
    ('classifier', LogisticRegression(random_state=42)),
])

lr_model_sol_comp1.fit(train1, train1.Estado_Final)

y_pred1 = lr_model_sol_comp1.predict(validation1)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


In [55]:

metrics.accuracy_score(validation1.Estado_Final, y_pred1)

0.8599629858112277

In [56]:
val_eval = validation1.copy()

# Probabilidad de pago
val_eval['prob_paga'] = lr_model_sol_comp1.predict_proba(validation1)[:, 1]
val_eval['pred_binaria'] = lr_model_sol_comp1.predict(validation1)

In [57]:

# Los que se aprueban, los ordenamos de menor a mayor, con el objetivo de otorgar credito a mas personas. En validacion
candidatos_val = val_eval[val_eval['pred_binaria'] == 1].sort_values(
    by='Monto_Solicitado',
    ascending=True
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_val = []

for idx, fila in candidatos_val.iterrows():
    monto = fila['Monto_Solicitado']
    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados_val.append(fila['ID_Solicitud'])
    else:
        break

val_eval['decision_final'] = val_eval['ID_Solicitud'].isin(ids_aprobados_val).astype(int)

# Metrica local
ganancia_caso = np.where(
    val_eval['Estado_Final'] == 1,
    val_eval['Monto_Solicitado'] * val_eval['Tasa_Interes'],
    -val_eval['Monto_Solicitado']
)

ganancia_total = (ganancia_caso * val_eval['decision_final']).sum()

print(f"Préstamos aprobados en validación: {len(ids_aprobados_val)}")
print(f"Monto prestado: ${monto_acumulado:,.2f} USD")
print(f"Ganancia Neta obtenida: ${ganancia_total:,.2f} USD")

Préstamos aprobados en validación: 762
Monto prestado: $1,997,700.00 USD
Ganancia Neta obtenida: $193,855.32 USD


In [58]:
# Filtramos los que el modelo aprueba pero por probabilidad y ordenamos por probabilidad de pago ascendente en Validacion
candidatos_val = val_eval[val_eval['prob_paga'] >= 0.5].sort_values(
    by='prob_paga',
    ascending=False
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_val = []

for idx, fila in candidatos_val.iterrows():
    monto = fila['Monto_Solicitado']
    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados_val.append(fila['ID_Solicitud'])
    else:
        continue

# Asignamos la decisión final (1 si fue aprobado, 0 en caso contrario)
val_eval['decision_final'] = val_eval['ID_Solicitud'].isin(ids_aprobados_val).astype(int)

# Metrica local
ganancia_caso = np.where(
    val_eval['Estado_Final'] == 1,
    val_eval['Monto_Solicitado'] * val_eval['Tasa_Interes'],
    -val_eval['Monto_Solicitado']
)

ganancia_total = (ganancia_caso * val_eval['decision_final']).sum()

print(f"Préstamos aprobados en validación: {len(ids_aprobados_val)}")
print(f"Monto prestado: ${monto_acumulado:,.2f} USD")
print(f"Ganancia Neta obtenida: ${ganancia_total:,.2f} USD")

Préstamos aprobados en validación: 105
Monto prestado: $1,999,900.00 USD
Ganancia Neta obtenida: $447,802.38 USD


In [59]:
test_sol_comp = df_sol_test.merge(comportamiento_resumen, on= 'ID_Usuario', how= 'left')
test_sol_comp.head()


,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,32417,27074,2300,0.2771,2596,Consumo,2026-01,2104.447500,15.833333
1,32418,28172,10900,0.3291,6905,Consumo,2026-01,21749.880455,3.136364
2,32419,27547,1300,0.2485,3707,Educacion,2026-01,1211.650435,23.043478
3,32420,27689,1600,0.2699,3314,Consumo,2026-01,7926.885455,8.000000
4,32421,26219,900,0.2317,3870,Educacion,2026-01,6129.496087,3.956522


In [60]:
#ya se elimno idusuario
test_sol_comp.drop(columns=['Fecha_Solicitud'], inplace=True)
test_sol_comp.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,32417,27074,2300,0.2771,2596,Consumo,2104.447500,15.833333
1,32418,28172,10900,0.3291,6905,Consumo,21749.880455,3.136364
2,32419,27547,1300,0.2485,3707,Educacion,1211.650435,23.043478
3,32420,27689,1600,0.2699,3314,Consumo,7926.885455,8.000000
4,32421,26219,900,0.2317,3870,Educacion,6129.496087,3.956522


In [62]:
#Predicion binaria en test
test_eval=df_sol_test.copy()
test_eval['pred_binaria'] = lr_model_sol_comp1.predict(test_sol_comp)

# 3. Filtramos los aprobados (pred_binaria == 1) y ordenamos por Monto_Solicitado ascendente
candidatos_test = test_eval[test_eval['pred_binaria'] == 1].sort_values(
    by='Monto_Solicitado',
    ascending=True
)

# 4. Asignamos el presupuesto con el bucle
PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados_test = []

for idx, fila in candidatos_test.iterrows():
    monto = fila['Monto_Solicitado']
    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados_test.append(fila['ID_Solicitud'])
    else:
        break

# 5. Creamos la columna final de decisión
test_eval['Decision_Aprobacion'] = test_eval['ID_Solicitud'].isin(ids_aprobados_test).astype(int)

# 6. Exportamos el archivo exacto para Kaggle
submission = test_eval[['ID_Solicitud', 'Decision_Aprobacion']]
submission.to_csv('submission_monto_ascendente_binario.csv', index=False)

In [63]:
#prediccion probabilidad pago test
test_eval_prob = test_sol_comp.copy()
test_eval_prob['prob_paga'] = lr_model_sol_comp1.predict_proba(test_sol_comp)[:, 1]

# 3. Filtramos candidatos con prob_paga >= 0.5 y ordenamos de MAYOR a MENOR probabilidad
candidatos_prob = test_eval_prob[test_eval_prob['prob_paga'] >= 0.5].sort_values(
    by='prob_paga',
    ascending=False
)

# 4. Asignación con presupuesto de 2M usando continue para optimizar el capital
PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados = []

for idx, fila in candidatos_prob.iterrows():
    monto = fila['Monto_Solicitado']
    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados.append(fila['ID_Solicitud'])
    else:
        continue

# 5. Creamos la columna exacta requerida por la competencia
test_eval_prob['Decision_Aprobacion'] = test_eval_prob['ID_Solicitud'].isin(ids_aprobados).astype(int)


# 6. Exportamos el CSV para enviar a Kaggle
submission_prob = test_eval_prob[['ID_Solicitud', 'Decision_Aprobacion']]
submission_prob.to_csv('submission_predict_proba.csv', index=False)

